# Llama 3 with QLoRA (FP4 Quantization)

This notebook demonstrates how to load Llama-3 using 4-bit quantization and prepare it for LoRA fine-tuning using the `peft`, `bitsandbytes`, and `transformers` libraries. This approach allows you to train and run large models on consumer GPUs.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct" # this is the model id for the model

# 1. Configure 4-bit quantization (FP4 / NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="fp4", # Or "nf4" which is generally recommended for QLoRA
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer and model in 4-bit... this might take a moment if downloading.")

# 2. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. Load Model with QLoRA config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Prepare model for QLoRA training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 5. Define LoRA Configuration
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
